# Anaconda

https://www.anaconda.com/products/individual

# Reference
[uproot documentation](https://uproot.readthedocs.io/en/latest/)

# File management and helper classes

You'll find a Python module uproot_io alongside this notebook that provides some functions to process the ROOT files such that you shouldn't need to worry about how things are stored by uproot.

This sets up two classes, Events and View. The Events class holds all of the information loaded from a ROOT file (example usage is given below), as well as a couple of functions to help with filtering information by event number and PDG code. There's also a make_sequential function you don't need to worry about, which handles some technicalities around event number uniqueness. Examples using these filtering functions are given below.

You'll want to have a look at the Event class in particular, as the `__init__` function's  self.* variables represent all of the information you'll have access to. You won't necessarily need all (or even most) of these variables, but it's worthwhile to become familar with what's available.

View (a synonym for wire plane) is a helper class that can take an Event object and extract a core subset of information from an event in a given wire plane (most relevant to the BSc project). Each event in a View object is a collection of all hit coordinates in x (common to all wire planes) and z (the 'depth' in the U, V or W view as appropriate), their corresponding ADC values and the true vertex location in the coordinates of the relevant wire plane.

An example event display using View is given below.

In [ ]:
from uproot_io import Events, View
import numpy as np

# Interaction type lookup table

interaction_dictionary is a simple lookup table that takes the numerical interaction type  from the Events class and converts it to a human readable string, e.g.

`   interaction_dictionary[3]`

returns

`   'CCQEL_MU_P_P_P'`

indicating that an interaction type code of 3 happens to represent a charged-current quasi-elastic interaction with a muon and three protons in the final state.

In [ ]:
# Interaction type lookup
import csv
interaction_dictionary = {}
with open('interactions.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        key = int(row.pop('Idx'))
        interaction = row.pop('Interaction')
        interaction_dictionary[key] = interaction

# Get data from tree

In [ ]:
events = Events("TrainingFile.root")

If you want to extract information from a particular event, you can use the filter_by_event function alluded to above. In particular, you can pass in the variable you want filtered, and the event number (or list of numbers) you want to focus on and you'll get back a version of that variable containing only information from those events you specified.

This will mostly be useful for debugging, or gaining an understanding of the content of a variable without being overwhelmed by one million values from the entire array, but do take care if working with filtered variables to ensure that, if you are working with multiple variables, that they are all filtered in the same way.

As an example, if you want to the drift coordinates (x) in the U plane for event 1, you can use the call below. Note that multiple arrays are returned in this case, because hits are separated into reconstructed PFOs, of which there are 8 in this event. Each PFO has a varying number of hits, including one case where there are no hits in the U view (the PFO was reconstructed from hits available in the V and W views, but it still has an empty entry for the U view).

In [ ]:
events.filter_by_event(events.reco_hits_x_u, 1)

You can also filter by PDG code. The idea is the same; you pass in the variable you want to filter and the PDG code or list of codes you want. In this case, we haven't set the `respect_sign` variable, and so it is assumed that you want positively and negatively signed versions of the specified particle, as indicated by the presence if negatively charged pions (-211) in addition to the positively charged pions (+211), which is all we specified.

In [ ]:
events.filter_by_pdg(events.mc_pdg, [211, 22])

# Extract hits in each view

Getting the hits from each view is quite simple, you need only pass in an Event object and specify the view you want. Now however, if you access a variable in the view and specify a specific event, e.g.

`view_w.z[1]`

you'll get back one array with all of the information for that event in the view of interest. All of the PFO details are lost, but this approach is very convenient for event displays (see below).

In [ ]:
view_u = View(events, "u")
view_v = View(events, "v")
view_w = View(events, "w")

In [ ]:
view_w.z[1]

# Saving plots and numpy arrays

Below is a simple utility function you can call whenever you want to save a plot, creating directories as needed and producing output both in png, jpg and in vector graphics format

In [ ]:
import os

def save_plot(fig, directory, filename):
    if not os.path.exists(directory):
        os.mkdir(directory)
    # note, if using multiple plots png crops the axes
    fig.savefig(f'{directory}/{filename}.png', bbox_inches='tight', dpi=200)
    fig.savefig(f'{directory}/{filename}.jpg', bbox_inches='tight', dpi=200)
    fig.savefig(f'{directory}/{filename}.svg', bbox_inches='tight', dpi=200)

You'll likely produce numpy arrays as either final results or intermediate steps, it's worth getting into the habit of saving at least some of these arrays. While you will often work with small subsets of the data when testing out new functions there will be times when you will need the larger datasets, and depending on the computational efficiency of your code, some of these steps may take quite a long time.

Python often has a reputation for being slow relative to languages like C++. This is often not actually true, and well written code, using core Python features like list comprehensions can be very fast (they're often compiled C code), but on large data sets, say 100,000 events or more, even very efficient code might take a minute or two, which will quickly get annoying in a debugging cycle, but for less Pythonic code with nested loops, this might turn into many hours, so for intermediate results in particular, you want to be able to run such steps once and then just load the result when you need to do some additional post-processing.

For this numpy.save("your_filename", your_array) and the corresponding your_array = numpy.load("your_filename") will be helpful. Test that these are working on small arrays early and then be sure to save large arrays as necessary, being careful to ensure you don't overwrite things you meant to keep.